# Lecture 7 · Notebook 3 — Agent memory: files vs. embeddings
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture07/lecture07_3_agent_memory_files_vs_embeddings.ipynb)

**Companion to [Chapter 5 of the video lecture](https://nlp-course-cuhksz.github.io/lecture07.html#p5-s60)** (slides 60–61, *Text embeddings*, and slide 67, *Agent memory as
files*). The lecture's conclusion: *files for the agent's own memory, plus a vector index as one more search tool when the collection
grows.* In this notebook you will:

1. embed sentences in English and Chinese and see **meaning as geometry** (slide 60),
2. see why modern embedders take an **instruction prefix** (slide 61),
3. build a tiny **file-based memory** with a `MEMORY.md` index and search it with `grep`, the way coding agents do (slide 67),
4. compare grep and embedding search on exact, paraphrased and cross-language queries, and on an **updated** fact, and
5. compute the **InfoNCE** training loss from slide 61 with in-batch negatives.

Runs on CPU in about 2 minutes; the multilingual embedder (`intfloat/multilingual-e5-small`) is about 470 MB.

In [ ]:
!pip -q install -U transformers

In [ ]:
import torch, torch.nn.functional as F, numpy as np, matplotlib.pyplot as plt, os, re, glob
from transformers import AutoTokenizer, AutoModel

MODEL_ID = "intfloat/multilingual-e5-small"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
enc = AutoModel.from_pretrained(MODEL_ID).eval()

@torch.no_grad()
def embed(texts, kind="passage"):
    # e5 models are trained with a prefix: "query: " for what you search with, "passage: " for what you store
    batch = tok([f"{kind}: {t}" for t in texts], padding=True, truncation=True, max_length=256, return_tensors="pt")
    h = enc(**batch).last_hidden_state
    mask = batch["attention_mask"].unsqueeze(-1).float()
    return F.normalize((h * mask).sum(1) / mask.sum(1), dim=-1)      # mean-pool, then length 1: cosine = dot product
print("embedding size:", embed(["hello"]).shape[1])

## 1 · Meaning as geometry
The example from slide 60: two sentences about a cat (one English, one Chinese), two about the stock market, and a query that never
says "cat" or "mat".

In [ ]:
docs = ["The cat is sleeping on the mat.", "猫在垫子上睡觉。", "The stock market fell sharply today.", "投资者在股价下跌时卖出了股票。"]
query = "Where is my kitten?"
D, q = embed(docs), embed([query], "query")
for d, s in zip(docs, (q @ D.T)[0]):
    print(f"{s:.3f}  {d}")
S = D @ D.T
plt.figure(figsize=(4.5, 3.5)); plt.imshow(S, cmap="Reds", vmin=0.6, vmax=1)
plt.xticks(range(4), ["cat EN", "cat ZH", "stock EN", "stock ZH"]); plt.yticks(range(4), ["cat EN", "cat ZH", "stock EN", "stock ZH"])
plt.colorbar(); plt.title("cosine similarity between documents"); plt.show()

The cat sentences rank above the stock sentences in both languages. Note that e5's cosines live in a narrow band (roughly 0.7–0.9):
for retrieval only the **ranking** matters, not the absolute number. Different embedders have different scales.

## 2 · The instruction prefix
Slide 61: modern embedders are told what kind of similarity we want. e5 uses `query:` / `passage:`. Compare the ranking with and
without the right prefix for a question-style query:

In [ ]:
corpus = ["Marvin Minsky published The Society of Mind in 1986.",
          "When was The Society of Mind published?",
          "The Society of Mind argues that intelligence comes from many simple agents."]
question = "When was The Society of Mind published?"
for kind in ["query", "passage"]:
    s = (embed([question], kind) @ embed(corpus).T)[0]
    print(f"question embedded as '{kind}:'")
    for i in s.argsort(descending=True):
        print(f"   {s[i]:.3f}  {corpus[i]}")

What you want for question answering is the passage that **answers** the question, not a copy of it. With the `query:` prefix the
copy is no longer identical (similarity below 1) and the answer moves closer, but this small model still ranks the copy first: a reason
to deduplicate your index and to let a reranker read the top candidates (slide 63). With other embedders (Qwen3-Embedding, BGE) the
prefix is a full instruction such as *"Given a question, retrieve passages that answer it"*.

## 3 · Memory as files
Slide 67: one small file per fact, plus an index `MEMORY.md` with one line per memory. The index is loaded every session; details
are found with `grep` or by opening the right file. The user "Alex" below is made up.

In [ ]:
MEM = "memory"
os.makedirs(MEM, exist_ok=True)
FACTS = {
    "pet.md": "Alex has a cat named Mochi who sleeps on the windowsill.",
    "allergy.md": "Alex is allergic to peanuts; never suggest recipes with peanuts.",
    "language.md": "Alex prefers answers in Chinese, with technical terms in English.",
    "hardware.md": "Alex's laptop has an RTX 4060 GPU with 8 GB of memory.",
    "experiment.md": "The baseline experiment uses learning rate 3e-4 and batch size 32.",
    "team.md": "Mei handles the data pipeline and Tom runs the training jobs.",
    "deadline.md": "The project report is due on Friday; the poster is due one week later.",
    "editor.md": "Alex uses VS Code with the Vim extension.",
}
for f, text in FACTS.items():
    open(os.path.join(MEM, f), "w", encoding="utf-8").write(text + "\n")
open(os.path.join(MEM, "MEMORY.md"), "w", encoding="utf-8").write("".join(f"- [{f}]({f}): {t[:50]}...\n" for f, t in FACTS.items()))
print(open(os.path.join(MEM, "MEMORY.md"), encoding="utf-8").read())

def grep(pattern):
    hits = []
    for path in sorted(glob.glob(os.path.join(MEM, "*.md"))):
        if path.endswith("MEMORY.md"): continue
        for line in open(path, encoding="utf-8"):
            if re.search(pattern, line, re.I): hits.append((os.path.basename(path), line.strip()))
    return hits

print(grep(r"learning rate"))

## 4 · grep vs. a vector index
Build a vector index over the same files (embed each file once, offline), then ask five questions. For grep we use what an agent would
type: the obvious keywords of the question.

In [ ]:
def build_index():
    files = sorted(f for f in os.listdir(MEM) if f != "MEMORY.md")
    texts = [open(os.path.join(MEM, f), encoding="utf-8").read().strip() for f in files]
    return files, texts, embed(texts)
files, texts, index = build_index()

def vsearch(question, k=1):
    s = (embed([question], "query") @ index.T)[0]
    top = s.topk(k)
    return [(files[i], texts[i], round(v.item(), 3)) for v, i in zip(top.values, top.indices)]

QUESTIONS = [                     # (question, the regex an agent would grep for, the file that answers it)
    ("What learning rate does the baseline use?", r"learning rate", "experiment.md"),
    ("Does Alex have any pets?", r"pet", "pet.md"),
    ("Which GPU is in Alex's laptop?", r"GPU", "hardware.md"),
    ("Alex 对什么食物过敏？", r"过敏", "allergy.md"),
    ("Who is responsible for training the models?", r"responsible|training", "team.md"),
]
for qtext, pattern, gold in QUESTIONS:
    g = grep(pattern)
    v = vsearch(qtext)[0]
    print(f"Q: {qtext}\n   grep /{pattern}/ -> {[h[0] for h in g] or 'no hits'}   {'OK' if any(h[0] == gold for h in g) else 'MISS'}"
          f"\n   vector top-1  -> {v[0]} ({v[2]})   {'OK' if v[0] == gold else 'MISS'}")

Typical pattern: grep wins on exact strings and identifiers (learning rates, GPU names, file paths) and is perfectly transparent; it
misses paraphrases ("pets" vs. "cat") and other languages. The embedding finds those. An agent can grep again with better words,
which is what coding agents do, but every retry costs a step.

### Updating a fact
Alex adopts a dog. Editing one file is all a file-based memory needs. A vector index built earlier is **stale** until you re-embed.

In [ ]:
open(os.path.join(MEM, "pet.md"), "w", encoding="utf-8").write("Alex has a cat named Mochi and, since May, a dog named Bao.\n")
print("grep now:     ", grep(r"dog"))
print("stale index:  ", vsearch("What is the name of Alex's dog?")[0])
files, texts, index = build_index()             # re-embed everything (cheap here, expensive for millions of documents)
print("fresh index:  ", vsearch("What is the name of Alex's dog?")[0])

The stale index still points to `pet.md`, but it serves the text it embedded earlier, which mentions no dog. In a file memory the
agent always reads the current file. That is the trade-off on slide 67: people can read and fix files, and updating a fact means
editing, not re-embedding; vectors win on fuzzy and cross-language matching and on scale.

## 5 · How an embedder is trained: InfoNCE with in-batch negatives
Slide 61, right side. Take B (query, positive) pairs, embed them, and build the B×B similarity matrix S divided by a temperature τ.
Row i is a classification problem: *which document is mine?* The right answer is column i; every other column is a free negative.
The loss is the cross-entropy with labels 0…B−1, the same loss as CLIP.

In [ ]:
pairs = [("Does Alex have any pets?", FACTS["pet.md"]), ("Which GPU does Alex have?", FACTS["hardware.md"]),
         ("What is the learning rate?", FACTS["experiment.md"]), ("Who runs training?", FACTS["team.md"]),
         ("When is the report due?", FACTS["deadline.md"]), ("Which editor does Alex use?", FACTS["editor.md"])]
Q = embed([a for a, _ in pairs], "query")
P = embed([b for _, b in pairs])
labels = torch.arange(len(pairs))
for tau in [1.0, 0.05, 0.01]:
    S = Q @ P.T / tau
    print(f"tau={tau:<5} loss = {F.cross_entropy(S, labels).item():.3f}   "
          f"P(own positive) = {S.softmax(-1).diag().mean().item():.2f}")
S_shuffled = Q @ P[torch.randperm(len(pairs), generator=torch.Generator().manual_seed(1))].T / 0.05
print("shuffled pairs, tau=0.05: loss =", round(F.cross_entropy(S_shuffled, labels).item(), 3))
plt.figure(figsize=(4, 3.5)); plt.imshow((Q @ P.T).numpy(), cmap="Reds"); plt.colorbar()
plt.xlabel("documents (positives of other rows = negatives)"); plt.ylabel("queries"); plt.title("S = Q Pᵀ"); plt.show()

With τ = 1 the softmax is almost flat, so the loss stays near log B even for a good model; a small τ (0.01–0.05, as on the slide)
sharpens it, so the model is rewarded for putting each query's own positive clearly on top. Shuffling the pairs makes the
loss large: the diagonal no longer holds the true matches.

## Exercises
1. **Hard negatives.** Add the document *"Alex's sister has a parrot."* to the index. Does "Does Alex have any pets?" still retrieve
   `pet.md`? This near-miss is what slide 61 calls a hard negative.
2. **Agent-style grep.** Write `smart_grep(question)` that asks the 0.5B model from Notebook 1 for three regex patterns and greps all
   of them. How many of the MISS rows does it fix, and at what cost in model calls?
3. **Shorter vectors.** Keep only the first 128 of the 384 dimensions (re-normalize!). Does the ranking in section 4 survive?
   (e5 was not trained with the Matryoshka trick from slide 61, so expect some damage.)

**Back:** [Notebook 2 — pass@k vs. pass^k]({colab(N2)}) · [the lecture page]({SITE})